## Normalize

In [ ]:
import json
import regex as re
import shutil
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Tuple

# from datasets import Audio, Dataset, DatasetDict, load_from_disk
from khmercut import tokenize

from khmerspeech.text_normalization import (
  normalize,
  datetime as km_datetime,
  phone_numbers,
  currency,
  cardinals,
  decimals,
  urls,
  ordinals,
  dict_verbalize,
  repeater,
  punctuations,
  parenthesis
)
from khmerspeech import TN, asr_normalize, text_normalization_gemini



## The problem for Text Normalize in Khmer language

#### Khmercut
$100.25 -> $100. 25 
https://google.com.kh -> https://google.com. kh

#### Khmerspeech
2024-01-02 -> ពីរពាន់▁ម្ភៃបួន មួយ ពីរ



In [4]:
texts = """
ការសិក្សាត្រូវមាន       ការព្យាយាម this  is the function for word segmentation!!! 
010123123 គិតថ្លៃ $100.25 នៅថ្ងៃទី 2024-01-02 វេលា 10:23AM ចូលតាម https://google.com.kh
ត្រូវនឹងថ្ងៃទី២២   ខែតុលា ឆ្នាំ២០១៩ ស្តីពីសិទ្ធិអំណាចក្នុងការដាក់ឲ្យប្រើប្រាស់ជាផ្លូវការ​នូវលិខិតបទដ្ឋានផ្ទៃក្នុង ធនាគារ អេស៊ីលីដា ភីអិលស៊ី ។
សេចក្តីសម្រេចលេខៈ ធ_អ_ក ១០១៦\១៩ ចុះថ្ងៃអង្គារ ៩រោច ខែអស្សុជ ឆ្នាំកុរ ឯកស័ក ព.ស.២៥៦៣
សេចក្តីសម្រេចលេខៈ ធអក-ប្រក១៤០៤\២២ ចុះថ្ងៃអង្គារ ៨កើត ខែកត្តិក ឆ្នាំខាល ចត្វាស័ក ព.ស២៥៦៦
យោង : សេចក្តីសម្រេចលេខៈ ធអក ១២៥៤\២៤ ចុះថ្ងៃចន្ទ ១៣រោច ខែភទ្របទ ឆ្នាំរោង ឆស័ក ព.ស.២៥៦៨
គណៈប្រធាននាយិកាប្រតិបត្តិប្រតិបត្តិការ ជម្រាបជូន  នាយកជាន់ខ្ពស់សាខា ខណ្ឌសែនសុខ-សង្កាត់អូរបែកក្អម
គណនេយ្យករឬភ្នាក់ងារប្រតិបត្តិការត្រូវធ្វើការលុបចោល(Waive តាមនីតិវិធីរបស់នាយកដ្ឋានហិរញ្ញវត្ថុ) ចំនួនការប្រាក់ដែលកើតឡើង
"""

In [5]:
def km_word_segment(text: str) -> str:
    tokens = tokenize(str(text))
    s = " ".join(tokens)
    return re.compile(r"\s+").sub(" ", s).strip()

segment_text = km_word_segment(texts)
print(segment_text)

ការសិក្សា ត្រូវ មាន ការព្យាយាម this is the function for word segmentation!!! 010123123 គិត ថ្លៃ $100. 25 នៅ ថ្ងៃទី 2024-01-02 វេលា 10:23AM ចូល តាម https://google.com. kh ត្រូវនឹង ថ្ងៃទី ២២ ខែតុលា ឆ្នាំ ២០១៩ ស្តីពី សិទ្ធិ អំណាច ក្នុង ការដាក់ ឲ្យ ប្រើប្រាស់ ជា ផ្លូវការ​ នូវ លិខិត បទដ្ឋាន ផ្ទៃក្នុង ធនាគារ អេស៊ីលីដា ភី អិល ស៊ី ។ សេចក្តីសម្រេច លេខៈ ធ_ អ_ក ១០១៦ \១៩ ចុះ ថ្ងៃអង្គារ ៩រោច ខែ អស្សុជ ឆ្នាំ កុរ ឯកស័ក ព .ស.២៥៦៣ សេចក្តីសម្រេច លេខៈ ធ អក - ប្រក ១៤០៤ \ ២២ ចុះ ថ្ងៃអង្គារ ៨ កើត ខែកត្តិក ឆ្នាំ ខាល ចត្វា ស័ក ព .ស ២៥៦៦ យោង : សេចក្តីសម្រេច លេខៈ ធ អក ១២៥៤\ ២៤ ចុះ ថ្ងៃចន្ទ ១៣រោច ខែ ភទ្រប ទ ឆ្នាំ រោង ឆស័ក ព .ស .២៥៦៨ គណៈ ប្រធាន នាយិកា ប្រតិបត្តិ ប្រតិបត្តិការ ជម្រាប ជូន នាយក ជាន់ ខ្ពស់ សាខា ខណ្ឌ សែនសុខ - សង្កាត់ អូរបែក ក្អម គណនេយ្យក រ ឬ ភ្នាក់ងារ ប្រតិបត្តិការ ត្រូវ ធ្វើ ការលុបចោល (Waive តាម នីតិវិធី របស់ នាយកដ្ឋាន ហិរញ្ញវត្ថុ ) ចំនួន ការប្រាក់ ដែល កើតឡើង


In [6]:
TN(segment_text)

'ការសិក្សា ត្រូវ មាន ការព្យាយាម this is the function for word segmentation ! សូន្យ▁ដប់▁មួយរយ▁ម្ភៃបី▁មួយរយ▁ម្ភៃបី គិត ថ្លៃ មួយរយ▁ដុល្លារចុច ម្ភៃប្រាំ នៅ ថ្ងៃទី ពីរពាន់▁ម្ភៃបួនត្រេមួយត្រេពីរ វេលា ដប់ចុចពីរម្ភៃបីAM ចូល តាម httpsចុចពីរស្លាសស្លាសgoogleចុចcomចុច kh ត្រូវនឹង ថ្ងៃទី ម្ភៃពីរ ខែតុលា ឆ្នាំ ពីរពាន់▁ដប់ប្រាំបួន ស្តីពី សិទ្ធិ អំណាច ក្នុង ការដាក់ ឱ្យ ប្រើប្រាស់ ជា ផ្លូវការ នូវ លិខិត បទដ្ឋាន ផ្ទៃក្នុង ធនាគារ អេស៊ីលីដា ភី អិល ស៊ី ។ សេចក្តីសម្រេច លេខៈ ធអាន់ឌើស្ករ អអាន់ឌើស្ករក មួយពាន់▁ដប់ប្រាំមួយ ស្លាសទៅក្រោយដប់ប្រាំបួន ចុះ ថ្ងៃអង្គារ ប្រាំបួនរោច ខែ អស្សុជ ឆ្នាំ កុរ ឯកស័ក ព ចុចសចុចពីរពាន់▁ប្រាំរយ▁ហុកសិបបី សេចក្តីសម្រេច លេខៈ ធ អក ត្រេ ប្រក មួយពាន់▁បួនរយ▁បួន ស្លាសទៅក្រោយ ម្ភៃពីរ ចុះ ថ្ងៃអង្គារ ប្រាំបី កើត ខែកត្តិក ឆ្នាំ ខាល ចត្វា ស័ក ព ចុចស ពីរពាន់▁ប្រាំរយ▁ហុកសិបប្រាំមួយ យោង ចុចពីរ សេចក្តីសម្រេច លេខៈ ធ អក មួយពាន់▁ពីររយ▁ហាសិបបួនស្លាសទៅក្រោយ ម្ភៃបួន ចុះ ថ្ងៃចន្ទ ដប់បីរោច ខែ ភទ្រប ទ ឆ្នាំ រោង ឆស័ក ព ចុចស ចុចពីរពាន់▁ប្រាំរយ▁ហុកសិបប្រាំបី គណៈ ប្រធាន នាយិកា ប្រតិបត្តិ ប្រតិបត្តិការ ជម្រាប ជូន នា

In [7]:
tn_texts = TN(texts)
tn_texts

'ការសិក្សាត្រូវមាន ការព្យាយាម this is the function for word segmentation ! \nសូន្យ▁ដប់▁មួយរយ▁ម្ភៃបី▁មួយរយ▁ម្ភៃបី គិតថ្លៃ មួយរយ▁ដុល្លារចុចម្ភៃប្រាំ នៅថ្ងៃទី ពីរពាន់▁ម្ភៃបួនត្រេមួយត្រេពីរ វេលា ដប់ចុចពីរម្ភៃបីAM ចូលតាម httpsចុចពីរស្លាសស្លាសgoogleចុចcomចុចkh\nត្រូវនឹងថ្ងៃទីម្ភៃពីរ ខែតុលា ឆ្នាំពីរពាន់▁ដប់ប្រាំបួន ស្តីពីសិទ្ធិអំណាចក្នុងការដាក់ឱ្យប្រើប្រាស់ជាផ្លូវការនូវលិខិតបទដ្ឋានផ្ទៃក្នុង ធនាគារ អេស៊ីលីដា ភីអិលស៊ី ។\nសេចក្តីសម្រេចលេខៈ ធអាន់ឌើស្ករអអាន់ឌើស្ករក មួយពាន់▁ដប់ប្រាំមួយស្លាសទៅក្រោយដប់ប្រាំបួន ចុះថ្ងៃអង្គារ ប្រាំបួនរោច ខែអស្សុជ ឆ្នាំកុរ ឯកស័ក ពចុចសចុចពីរពាន់▁ប្រាំរយ▁ហុកសិបបី\nសេចក្តីសម្រេចលេខៈ ធ_អ_កត្រេប្រកមួយពាន់▁បួនរយ▁បួនស្លាសទៅក្រោយម្ភៃពីរ ចុះថ្ងៃអង្គារ ប្រាំបីកើត ខែកត្តិក ឆ្នាំខាល ចត្វាស័ក ពចុចសពីរពាន់▁ប្រាំរយ▁ហុកសិបប្រាំមួយ\nយោង ចុចពីរ សេចក្តីសម្រេចលេខៈ ធ_អ_ក មួយពាន់▁ពីររយ▁ហាសិបបួនស្លាសទៅក្រោយម្ភៃបួន ចុះថ្ងៃចន្ទ ដប់បីរោច ខែភទ្របទ ឆ្នាំរោង ឆស័ក ពចុចសចុចពីរពាន់▁ប្រាំរយ▁ហុកសិបប្រាំបី\nគណៈប្រធាននាយិកាប្រតិបត្តិប្រតិបត្តិការ ជម្រាបជូន នាយកជាន់ខ្ពស់សាខា ខណ្ឌសែនសុខត្រេសង្កាត់អូរបែកក្អម

In [8]:
km_word_segment(tn_texts)

'ការសិក្សា ត្រូវ មាន ការព្យាយាម this is the function for word segmentation ! សូន្យ ▁ ដប់ ▁ មួយ រយ ▁ម្ភៃ បី ▁ មួយ រយ ▁ម្ភៃ បី គិត ថ្លៃ មួយ រយ ▁ ដុល្លារ ចុច ម្ភៃ ប្រាំ នៅ ថ្ងៃទី ពីរ ពាន់ ▁ម្ភៃ បួន ត្រេ មួយ ត្រេ ពីរ វេលា ដប់ ចុច ពីរ ម្ភៃ បី AM ចូល តាម https ចុច ពីរ ស្លាស ស្លាស google ចុច com ចុច kh ត្រូវនឹង ថ្ងៃទី ម្ភៃ ពីរ ខែតុលា ឆ្នាំ ពីរ ពាន់ ▁ ដប់ ប្រាំបួន ស្តីពី សិទ្ធិ អំណាច ក្នុង ការដាក់ ឱ្យ ប្រើប្រាស់ ជា ផ្លូវការ នូវ លិខិត បទដ្ឋាន ផ្ទៃក្នុង ធនាគារ អេស៊ីលីដា ភី អិល ស៊ី ។ សេចក្តីសម្រេច លេខៈ ធ អាន់ ឌើ ស្ករ អអាន់ ឌើ ស្ករ ក មួយ ពាន់ ▁ ដប់ ប្រាំមួយ ស្លាស ទៅ ក្រោយ ដប់ ប្រាំបួន ចុះ ថ្ងៃអង្គារ ប្រាំបួន រោច ខែ អស្សុជ ឆ្នាំ កុរ ឯកស័ក ព ចុច សចុច ពីរ ពាន់ ▁ ប្រាំរយ ▁ ហុកសិប បី សេចក្តីសម្រេច លេខៈ ធ_ អ_កត្រេ ប្រក មួយ ពាន់ ▁ បួន រយ ▁ បួន ស្លាស ទៅ ក្រោយ ម្ភៃ ពីរ ចុះ ថ្ងៃអង្គារ ប្រាំបី កើត ខែកត្តិក ឆ្នាំ ខាល ចត្វា ស័ក ព ចុច ស ពីរ ពាន់ ▁ ប្រាំរយ ▁ ហុកសិប ប្រាំមួយ យោង ចុច ពីរ សេចក្តីសម្រេច លេខៈ ធ_ អ_ក មួយ ពាន់ ▁ ពីរ រយ ▁ ហាសិប បួន ស្លាស ទៅ ក្រោយ ម្ភៃ បួន ចុះ ថ្ងៃចន្ទ ដប់ បី រោច ខែ ភទ្រប ទ ឆ្នាំ រោង ឆស័

In [9]:
# asr_normalize(texts)

## Phone Numbers

In [10]:
from khmerspeech.text_normalization import phone_numbers

phone_numbers.processor("010123123", chunk_size=2)

'0▁10▁12▁31▁23'

In [ ]:
phone_numbers.processor("010123123", chunk_size=3)

'0▁10▁123▁123'

## URLs and Emails

In [ ]:
from khmerspeech.text_normalization import urls

text = """
Contact us at support@gmail.com or visit http://moe.gov.kh for more info.
Local test: http://localhost:8080.
Acleda : https://www.acledabank.com.kh/
"""

text2 = "http://localhost:2000   https://www.acledabank.com.kh http://localhost:2000"

print(urls.processor(text2))





has url
no url
host.lower() port 2000   www dot acledabank dot com dot k▁h host.lower() port 2000


## hashtags

## repeaters

## Datetimes

In [ ]:

from khmerspeech.text_normalization import datetime

# (your regexes and functions here...)

text = (
    "Meet on 2025-10-20 at 7:05pm.\n"
    "ពិភាក្សា ២០២៥/១០/២០ ម៉ោង ០៨:៥៩ AM.\n"
    "Invoice date: 20/10/2025 and follow-up at 12:30."
)

print("Original:")
print(text)

print("\nDates normalized:")
print(datetime.date_processor(text))

print("\nTimes normalized:")
print(datetime.time_processor(text))

print("\nDates then Times (combined):")
print(datetime.time_processor(datetime.date_processor(text)))


Original:
Meet on 2025-10-20 at 7:05pm.
ពិភាក្សា ២០២៥/១០/២០ ម៉ោង ០៨:៥៩ AM.
Invoice date: 20/10/2025 and follow-up at 12:30.

Dates normalized:
Meet on 2025 10 20 at 7:05pm.
ពិភាក្សា ២០២៥ ១០ ២០ ម៉ោង ០៨:៥៩ AM.
Invoice date: 20 10 2025 and follow-up at 12:30.

Times normalized:
Meet on 2025-10-20 at 7 05▁p▁m.
ពិភាក្សា ២០២៥/១០/២០ ម៉ោង ០៨ ៥៩▁A▁M.
Invoice date: 20/10/2025 and follow-up at 12 30.

Dates then Times (combined):
Meet on 2025 10 20 at 7 05▁p▁m.
ពិភាក្សា ២០២៥ ១០ ២០ ម៉ោង ០៨ ៥៩▁A▁M.
Invoice date: 20 10 2025 and follow-up at 12 30.


In [ ]:
import regex as re

RE_HASHTAGS = re.compile(r"\B(\#[a-zA-Z]+\b)")
RE_ENCLOSED_PARENTHESIS_BACKETS = re.compile(r"\s*\((.*?)\)\s*|\s*\[(.*?)\]\s*")
RE_GENERIC_NUMBER = re.compile(r"\d+\.?\d*")
RE_NUMBER_COMMAS = re.compile(r"(\d+)\,(\d+)")
RE_USELESS_COMMAS = re.compile(r"\s*\,\s*")
RE_ORDINAL_NUMBER = re.compile(r"^([\d\u17e0-\u17e9]+)\.\s")
RE_SPACING_COMMAS = re.compile(
  r"([^0-9\u17e0-\u17e9])[,]+([^0-9\u17e0-\u17e9]|$)"
)  # abc,abc

RE_NUM_LEADING_ZEROS = re.compile(r"\.([\u17e00]+)([\d\u17e0-\u17e9]+)")

def leading_zeros_replacer(m):
  padded_zeros = "▁".join(m[1])
  return f".▁{padded_zeros}▁{m[2]}"



In [ ]:
"1.23$" = "10 dollar 25 cent"


គ ស 

នីមួយៗ --> នីមួយ 